# 0. Imports and defaults

As you work through this notebook, you will have to modify modules `model.py` and `preprocessing.py`. Whenever you do so, remember to: 
- Save the two modules after modifying them, then come back to this notebook;
- Restart the kernel in this notebook ->  ; 
- Rerun all cells in this section (0. Imports and defaults). 

This ensures that this notebook "sees" all the changes you implemented in the external modules and updates the corresponding function definitions. 

In [1]:
import pandas as pd
import numpy as np
import pylab as plt

import pypsa

# Import the public function from the package you completed
from energy_model import model_run

In [2]:
# Default parameters for testing

# input database folder
db_dir = 'input_databases//default'

# model years
years = [2020,2030,2040,2050]

# intra-year temporal resolution
t_res = 15 # hours

# discount rates
rt = 0.07 # technical discount rate
rs = 0.01 # social discount rate

# passenger cars conversion factor
passenger_cars_cf = 40*0.1  # km/h


In [3]:
# Handy plotting functions

def plot_main_ebs(scen):
    fig, ax = plt.subplots(1,5)
    fig.set_size_inches(20,4)

    for n,c in enumerate(['Residential electricity',
            'Residential heat',
            'Transportation',
            'Industrial machinery',
            'Industrial HTP']): 
        df = scen.statistics.energy_balance(bus_carrier=c)
        df = df.set_index(df.index.get_level_values(1))
        df.T.plot(kind='bar',stacked='True', rot=0, colormap='tab20',ax=ax[n])
        if c=='Transportation':
            ax[n].set_ylabel('Mvkm')
        else:
            ax[n].set_ylabel('MWh')
        ax[n].set_title(c)

    fig.tight_layout()

def plot_elc_emi(scen):
    fig, ax = plt.subplots(1,5)
    fig.set_size_inches(20,4)

    for n,c in enumerate(['Electricity','CO2 atmosphere', 'CO2 storage']):
        df = scen.statistics.energy_balance(bus_carrier=c)
        df = df.set_index(df.index.get_level_values(1))
        df.T.plot(kind='bar',stacked='True', rot=0, colormap='tab20', ax=ax[n])

        if 'CO2' in c:
            ax[n].set_ylabel('tCO2')
        else:
            ax[n].set_ylabel('MWh')
        ax[n].set_title(c)
        ax[n].legend(bbox_to_anchor=(1,-0.5))

    # Let's add cumulative CO2
    for n,c in enumerate(['CO2 atmosphere', 'CO2 storage']):
        df = scen.statistics.energy_balance(bus_carrier=c).cumsum(axis=1)
        df = df.set_index(df.index.get_level_values(1))
        df.T.plot(kind='bar', stacked=True, colormap='tab20', ax=ax[n+3])
        ax[n+3].set_ylabel('tCO2')
        ax[n+3].set_title(f'Cumulative {c}')
        ax[n+3].legend(bbox_to_anchor=(1,-0.5))

    fig.tight_layout()


# 1. A beefed-up model
The code in ```main.py``` is exactly the same as last week, but this time several resources, demand segments and technologies have been added to the input database, following what we learned in the last lesson, when analyzing RES demand and supply:

- **Demand**
    - *residential_heat*: Combined demand for space and water heating in households
    - *industrial_machinery*: Generic demand for industrial machinery (assumed to be 40% of total industrial demand); machinery can easily be electrified
    - *industrial_htp*: Generic demand for industrial high temperature processes (assumed to be 60% of total industrial demand); high temperature processes are hard to abate and cannot be electrified

- **Supply**
    - *Biomass supply*: Generic bio-based reserve, it acts as a fossil fuel but has net 0 emissions
    - *Nuclear supply*: Uraniuim reserves for use in nuclar reactors
    - *CO2 storage*: A reserve that is distinct from CO2 atmosphere, and represents CO2 that is captured via CCS processes and stored underground; like all reserves it can be bound to simulate limited availability of storage sites, and it can be associated with a price (CO2 storage price)

- **Technologies**
    - Power sector technologies: 
        - *Nuclear plant*: Traditional, large scale, nuclear facility
        - *Biomass plant*: Same as in last week's homework
        - *Biomass plant CCS*: Bio-CCS (aka BECCS) delivers net negative emissions
        - *CCGT CCS*: GGCT plant equipped with CO2 capture with 85% capture efficiency in 2020 growing to 95% in 2050

    - Residential end-use technologies:
        - *Residential gas boiler*: Traditional gas-based boiler to generate hot water and space heating
        - *Heat pump*: Heat pump to provide hot water and space heating using electricity
    
    - Industrial end-use technologies: 
        - *Industrial machinery - fossil*: Generic industrial machinery running on fossil energy (we assume gas for simplicity)
        - *Industrial machinery - fossil CCS*: Same as above but equipped with CCS
        - *Industrial machinery - electricity*: Electrification as main decarbonization option for industrial machinery
        - *Industrial HTP - fossil*: Generic industrial HTP running on fossil energy (we assume gas for simplicity)
        - *Industrial HTP - fossil CCS*: Same as above but equipped with CCS

Additionally, the `technology.xlsx` file contains a new tab called *capacity_constraints*, which we will use for model calibration. For now this tab is basically empty: we will fill it in in the course of this practical. 

Spend a few minutes verifying that you can find these items in the database, and checking all associated entries in the various tabs. 
        
We start by running the model and analyzing its energy balances in this new setup.

In [4]:
base = model_run(db_dir,years,t_res,passenger_cars_cf,rt,rs)
base.name = 'base'

2020
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 406.50it/s]


Set parameter Username
Set parameter LicenseID to value 2858749
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file /private/var/folders/fx/w8lwd2d95938qx99fyfp5kw00000gp/T/linopy-problem-dajgqfb8.lp
Reading time = 0.42 seconds
obj: 233076 rows, 106932 columns, 484028 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2524053566659.803


In [ ]:
# Let's have a quick look at the demand-side energy balances:
plot_main_ebs(base)

In [ ]:
# Let's also look at electricity supply and emissions
plot_elc_emi(base)

As solar PV and wind energy become cheaper, from 2040 electricity naturally starts taking over large portions of passenger transportation, residential heat and industrial machinery. Industrial HTP remains fully fossil-fuel-based. Emissions naturally decrease, and in 2050 they are mostly due to industrial activity. The CO2 budget of 1 GtCO2 is not fully utilized.  

There are a couple of things happening in the model: 
- As wind and solar become cheaper, so does the average electricity price;
- Gas reserves ($10^9$ MWh) are fully depleted;
- As gas resources deplete its (shadow) price increases;
- Gasoline and CO2 reserves (resp $10^9$ MWh and 1 GtCO2) are by far not fully utilized, hence their (shadow) price remains 0.

In [ ]:
# Let's show that gas reserves are fully depleted by 2050, while gasoline reserves are not:
fig, ax = plt.subplots(3,1)
fig.set_size_inches(10,10)
reserves = {'Gas':1e9, 'Gasoline':1e9, 'CO2 atmosphere':1e9}
for n,c in enumerate(['Gas', 'Gasoline','CO2 atmosphere']):
    base.statistics.energy_balance(bus_carrier=c).cumsum(axis=1).T.plot(kind='bar', stacked=True, ax=ax[n])
    ax[n].hlines(reserves[c],-0.5,4.5, label='Reserves', ls='--', color='black')
    ax[n].legend(bbox_to_anchor=(1,1))
    ax[n].set_title(c)
plt.tight_layout()

In [ ]:
# Show shadow prices of electricity, gas, gasoline and CO2 atmosphere
base.buses_t.marginal_price.groupby(level=0).agg('mean')[['Electricity','Gas supply','Gasoline supply','CO2 atmosphere']]

# 2. Missing links
You might have noticed that there are no CCS technologies in the model slution. Actually, there are no CCS technology in the model at all: 

In [ ]:
[link for link in base.links.index if 'CCS' in link]

This is because our code only adds to the network links with 2 or 3 buses, but CCS links have 4 buses:
- bus0: input
- bus1: main output
- bus2: emissions to the atmosphere (not captured), *CO2 atmosphere*
- bus3: emissions captured and stored, *CO2 stored* 

For CCS technologies to be available to our model, we need to add a block of code that processes such links. 

Open ```model.py``` and add the relevant code to add a 4-buses link to the network. 

Restart the kernel and rerun the cells in section 0, then come back to this point and proceed with the notebook. 

First, we verify that CCS technologies are now available to the model: 

In [ ]:
baseCCS = model_run(db_dir,years,t_res,passenger_cars_cf,rt,rs)
baseCCS.name = 'baseCCS'

In [ ]:
plot_main_ebs(baseCCS)
plot_elc_emi(baseCCS)
print([link for link in baseCCS.links.index if 'CCS' in link])

Good! The model now contains all the CCS processes, although they are not utilized in the solution (not needed, since they are more expensive than their non-ccs counterparts, and the CO2 budget is not stringent).

### Exercise: A towaeds a zero carbon budget
To verify that a system achieving a zero-carbon budget is theoretically possible, set the CO2 atmosphere Budget for the year 2050 to 0 in the resources_parameters table and run a new scenario.

In [ ]:
bzero = ### TODO
bzero.name = 'Zero Budget'

In [ ]:
plot_main_ebs(bzero)
plot_elc_emi(bzero)


Remarks:
- Note how the model deploys massive amounts of BECCS in 2040 to clean-up emissions from industrial HTP. In 2050 most (all?) industrial HTP fossil processes reach the end of their lifetime and the model replaces them with CCS processes to reduce emissions. Net CO2 emissions to the atmosphere go to 0 in 2050, and the rest of the emission cumulated over the whole 2020-2050 period are captured and stored via BECCS.
- The carbon budget is implemented in this model as a budget for the whole model horizon. However this budget can be changed in future years. E.g., we start in 2020 with a budget of 1 Gt CO2, to make sure the current system configuration doesn't lead to infeasibilities, then we can adjust the budget down in 2030, 2040, and so on. In this example we are reducing the budget at every period, reaching 0 in 2050. The model can then still emit CO2 in the previous year, but has to compensate later with negative emissions to achieve a net balance equal to 0 in 2050. 
- Now that CO2 atmosphere is restricted, the model will assign a price to this resource, i.e. a CO2 price. 



In [ ]:
# Show shadow prices of electricity, gas, gasoline and CO2 atmosphere
# Notice that we now indeed have a non-0 CO2 price!
bzero.buses_t.marginal_price.groupby(level=0).agg('mean')[['Electricity','Gas supply','Gasoline supply','CO2 atmosphere']]

# 3. Model calibration
Look at the bars for the year 2020 in the plots above: they are completely out of sync with the actual historical values! 

In this final section we will calibrate the model. The main objectives of the calibration are:
- Bring the base year energy mix as close as possible to statistics (we will allow ourselves to be pragmatic here)
- Ensure that the 2030 mix is not completely unrealistic
- Give the model the chance to decommission existing technologies and replace them with new ones during the first part of the horizon

The model we developed amounts to about 70% of the total final energy consumption in the Netherlands in 2020 (you can verify this by digging in the Eurostat data). Therefore, when calibrating the model to statistics, we will apply a factor of 0.7 (e.g. total solar PV capacity in 2020 was about 11 GW, we will calibrate our model to 11 x 0.7 = 7.8 GW).

We'll proceed in 3 steps: 
1. Add a number of "existing" technologies to the database
2. Fill in the capacity_constraints tab
3. Add code to read in and apply the capacity constraints, and handle existing technologies

# 3.1 Existing technologies
We will add 6 existing technologies to the model. They are all links, we don't need existing technologies for generators. You can find their static and dynamic parameter assumptions in file *existing_techs.xlsx*. These techonologies are basically copies of the corresponding technologies we already have in the model, but their CAPEX is set to 0 and their lifetime is reduced to 4 years. Later we will modify the code so that the model is only allowed to install these technologies in the base year. 

The existing technologies help us with the model calibration in two ways: 
- Since their CAPEX is 0 the model will always chose them whenever they are available, but we can constrain their capacities in the base year to match statistics.
- In real life existing technologies in 2020 will have a distribution of remaining lifetimes (i.e. some will be old and about to reach the end of their lifetime, while others will be relatively new). By setting a relatively short lifetime for existing technologies in our database, we can simulate their retirement. Ideally the lifetime we set should represent the center of the lifetimes distribution in the base year, so as to capture the average remaining lifetime observed in reality. Accurately tuning this parameter turns out to be a tedious and intricate task, hence, for simplicity we will just assume that this is 4 years for all existing technologies. Of course you are free to experiment and refine this estimate if it is important for your scenario setup, or include additional existing technologies to simulate several batches of retirement. 

To proceed, simply copy-paste these technologies into the right places in the input database. 

## 3.2 Capacity constraints
Next we need to set some constraints on the installed capacities of both existing and new technologies in the base year. To do this we will fill in the data in tab *capacity_constraints*. We will fill in the same value for max and min in 2020, so as to calibrate the model to that exact value. 

**Renewables in the power sector**

Navigate to this Eurostat table: https://ec.europa.eu/eurostat/databrowser/bookmark/b592a80a-d8ec-4f04-9f30-d03173605405?lang=en&createdAt=2026-08-05T13:46:18Z. It contains installed renewable capacity in the Dutch power sector for 2020. Retrieve capacities for: 
- Solar PV
- Wind onshore
- Wind offshore
- Total biofuels (solid biofuels + biogases + waste) for calibrating biomass plants

Fill in these values in the *capacity_constraints* tab, multiplying by 0.7 as discussed above to account for the fact that we are providing elecricity to fulfil only 70% of the total final demand. You can do the calculation directly in the excel sheet using the column "adjustment_factor". 

**Gas in the power sector**

We will attribute the rest of the power sector capacity to gas. Navigate to https://ec.europa.eu/eurostat/databrowser/bookmark/30ea2bfb-4756-4493-9f0e-ae7e6ffcac9d?lang=en&createdAt=2026-08-05T14:01:00Z and use the entry for "combustible fuels" to calibrate our CCGT technology. Use the 0.7 factor, and subtract the biomass plant capacity (which is also a combustible fuel).

**Nuclear energy**

Use the same Eurostat table to calibrate nuclear capacity. 

**Cars**

Check the total number of cars and the number of electric cars here: https://ec.europa.eu/eurostat/databrowser/bookmark/4a7dd8ed-ad1b-4f38-8198-3f2bde55f9e3?lang=en&createdAt=2026-08-05T14:27:40Z. Use the values to fill in the corresponding existing capacities for ICE cars and BEVs. Start with BEVs: convert the number in the table into millions, and fill it into the table using our passenger_cars_cf as adjustment factor to get the right unit of capacity. For ICE cars follow the same procedure using the total number of cars and subtracting the BEV capacity. (Note that for cars we are modelling the whole fleet, so no need to use the 0.7 correction factor!) 

**Capacity constraints in future years**

You will notice that you can fill in capacity constraints also for 2030, 2040 and so on. These can be used to tune the min/max capacity of *a particular vintage* of the corresponding technology. For the moment we will leave these empty. Later you can use them to tune the behavior of the model and create more realistic deployment and substitution trends in the solution. 



## 3.3 Code
We need to add in the code a function to read-in the capacity constraints:

In [ ]:
# 1. Paste the following  code in the preprocessing.py module, 
#    right at the end of function preprocess_technologies():

cn = pd.read_excel(f"input_database//technologies.xlsx", sheet_name='capacity_constraints', index_col=[0,1])
cn = cn.sort_index()

# Return cn (along with techs_s and techs_d) at the end of the function!

# 2. In model.py, go to function model_run() section 1. PREPROCESS INPUT DATA
#    read in cn along with techs_s and techs_d. 

# 3. In model.py, go to the MODEL HELPER FUNCTIONS section and 
#    add the following function to retrieve all new capacity constraints:
def get_cap_constraints(t, years, cn):
    """
    Returns a df with any user-defined constraints (cn) on capacity for tech t. 
    The base year is only used to set up fixed capacity bounds. This is not extrapolated to future years.
    Any min/max constraints set for future years are inter/extrapolated. 
    Returns a NaN df if there are no constraints for t. 
    """

    # Set up empty df
    df = pd.DataFrame(columns=['Technology','Type']+years)
    df.Technology = [t] * 2
    df.Type = ['min','max']
    df = df.set_index(['Technology','Type'])

    # Fill it with user-defined constraints from the technologies input table
    df.loc[:,:] = cn.loc[:,:]

    # The base year is only used to set up fixed capacity bounds, we don't want to extrapolate those to future years. 
    # We only want to int/extrapolate if there are user-constraints on max or min cap limits after the base year:
    # Slice df for years > base year, and store index of non-nan values
    df_idx = df.loc[:,[y for y in years if ((y>years[0]) and (y in df.columns))]]
    idx = np.where(df_idx.notna())

    # If there are no non-nan values, return df as it is:
    if len(df_idx.iloc[idx[0],idx[1]])==0:
        ret = df

    # otherwise int-extrapolate:
    else:
        ret = int_extrapolate(df, years)            

    return ret

Next we need to apply the capacity constraints to generators and links, using the p_nom_max/min parameters:

In [ ]:
# For both generators and links, after the initiating the technology loop, i.e. ```for t in techs_by_type(...)```                        
# add this line of code to retrieve the capacity constraints
cap_constraints = get_cap_constraints(t,years,cn)


# Next in the block of code that starts with ```n.add('Generator'/'Link')```, apply the capacity constraints using the following syntax:

# For generators
p_nom_max = cap_constraints.at[(t,'max'),y],
p_nom_min = cap_constraints.at[(t,'min'),y],

# For links
p_nom_max = cap_constraints.at[(t,'max'),y] / techs_d.at[(t,'Efficiency'), y],
p_nom_min = cap_constraints.at[(t,'min'),y] / techs_d.at[(t,'Efficiency'), y],



Finally we need a condition to check wether a link is existing, in which case we limit its availability to the base-year:

In [ ]:
# Modify the beginning of the code block that adds links to the model with the following if... else statement:

for t in techs_by_type('Link'):

    if (('existing' in t) and (y>years[0])):  # if t is labeled as "existing" and the current year in the loop (y) is not the base year,
        continue                              # move on to the next technology
    else:                                     # if t is not "existing" or y = base year add t to the model with the usual code
        
        # rest of the link code block here
        # ...

# When doing this, watch out that the whole code block for the links is correctly indented!!

## 3.4 Testing
Save model.py and preprocessing.py, restart the kernel and rerun the cells in section 0 of this notebook. 

Now run a new base, check energy balances and emissions and check the optimal_capacity table to verify that the capacity constraints have been applied. 

In [ ]:
baseCAL = # TODO
baseCAL.name = 'baseCAL'

In [ ]:
# Check the main energy balances:
plot_main_ebs(baseCAL)
plot_elc_emi(baseCAL)

In [ ]:
# You can run the following code to verify that your capactity constraints are respected:
for c in baseCAL.buses.carrier.unique():
    print('\n',c)
    display(baseCAL.statistics.optimal_capacity(bus_carrier=c))

# 4. What's next?

The model is now complete, with a basic calibration. Good job! :)

## 4.1 Play time! 

Take some time now to understand how the model behaves when you change some input parameters. Make a copy of the database and try to play around with the inputs. Some suggestions: 
- Change the CO2 price;
- Implement a carbon budget;
- Run the model every 5 years and/or change t_res;
- Set a minimum capactity requirement for biomass in the power sector; 
- Change the costs of a technology; 
- ...

Inspect the energy balance plots. Is the model always "well behaved", or do you see unrealistically high deployment of certain processes, or instances in which you get a so-called *bang-bang* solution (i.e. a technology being completely replaced by another in the next period)? If you see this type of "bad behavior", try to use the capacity constraints to tune the deployment rates and the substitution rates to represent more plausible trends. 

(TIP: At high carbon prices or stringent carbon budgets, the model will try to install very large renewables capacities in the power sector already in 2030 - typically representing growth rates that are not achievable in the real world. You can set max capacity constraints for the 2030 vintages of solar and wind to limit this growth. Note that if you set too stringent constraints the model may become unfeasible!)

## 4.2 Group modelling assignment

Once you are familiar with the model, you can start implementing your own scenarios, from the policy scenario document. Think about the following items, and plan your workflow for the coming two weeks:

- Do you need to extend the model topology in order to run your scenarios?
    - *Make a plan and do some preliminary tests*
    - *If you need to make changes, it is good practice to implement them in stages, and test each stage before moving to the next*

- Which parameters in the database do you need to change in your scenarios? 
    - *It's often a good idea to create copies of the database, each representing one scenario*

- Which model outputs do you need to retreive for your analysis? 
    - *Perhaps you can already write some functions to gather, aggregate and plot these outputs*
